In [ ]:
first_name = "Gregory"
last_name = "Miller"
email = "millegre001@tamu.edu"

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import itertools
from scipy.optimize import linprog
import pickle
import random
import joblib
from sklearn.model_selection import train_test_split
import pandas as pd
from collections import defaultdict
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

In [ ]:
#The function below will create the G matrix given the parameters n and k where it creates a k size identity matrix 
#and appends a k x n-k -100 to 100 random real numbers

In [ ]:
def G_generator(n, k):
    G = np.identity(k)
    matrix = np.random.uniform(-100, 100, (k, n - k)).astype(np.float32)
    G = np.concatenate((G, matrix), axis=1)
    #print(G)
    return G

In [ ]:
#The function below will make the shape to the max size and flatten it.

In [ ]:
def pad_G(G, max_n = 10, max_k = 6):
    k,n = G.shape
    padded = np.zeros((max_k, max_n), dtype=G.dtype)
    padded[:k,:n] = G
    return padded.flatten()

In [ ]:
#The function below is the tuple generator. 

In [ ]:
def Tuple_generator(n, m):
    tuples = []
    elements = list(range(n))
    for a in elements:
        for b in elements:
            if a != b:
                remaining = [x for x in elements if x not in {a,b}]
                X_sets = itertools.combinations(remaining, m-1) if m - 1 > 0 else [()]
                for X in X_sets:
                    psi_values = itertools.product([-1, 1], repeat = m)
                    for psi in psi_values:
                        tuples.append((a,b,X,psi))
    return tuples

In [ ]:
#The below follows the Linear progression file shown in the project pdf

In [ ]:
def LP_solver(G, a, b, X, psi):
    k, n = G.shape
    bounds = [(None, None)] * k

    X_sorted = sorted(X)
    Y = [x for x in range(n) if x not in {a,b} and x not in X_sorted]
    Y_sorted = sorted(Y)
    x_values = [a] + X_sorted + [b] + Y_sorted
    tau_inv = {val: i for i, val in enumerate(x_values)}
    #negated for minimization as used by the scipy.linprog
    c = [(-psi[0] * G[i, a]) for i in range(k)]
    c = np.array(c)

    A_ub = []
    b_ub = []
    
    for j in X_sorted:
        row = [(psi[tau_inv[j]] * G[i,j] - psi[0] * G[i, a]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(0)

    for j in X_sorted:
        row = [(-psi[tau_inv[j]] * G[i, j]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(-1)

    A_eq = [(G[i,b]) for i in range(k)]
    b_eq = [1]

    for j in Y_sorted:
        row = [(G[i, j]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(1)

    for j in Y_sorted:
        row = [(-G[i,j]) for i in range(k)]
        A_ub.append(row)
        b_ub.append(1)

    
    A_ub = np.array(A_ub)
    b_ub = np.array(b_ub)
    A_eq = np.array(A_eq).reshape(1,k)
    b_eq = np.array(b_eq)
    #print("shape of c ", c.shape)
    #print("shape of A_ub ", A_ub.shape)
    #print("shape of b_ub ", b_ub.shape)
    #print("shape of A_eq ", A_eq.shape)
    #print("shape of b_eq ", b_eq.shape)
    
    res = linprog(c, A_ub = A_ub, b_ub = b_ub, A_eq = A_eq, b_eq = b_eq, bounds = bounds, method = 'highs')

    #print(f"Tuple: a={a}, b={b}, X={X}, psi={psi}")
    #print(f"Objective value (raw): {-res.fun if res.success else 'N/A'}")
    #print(f"Solver success: {res.success}, status: {res.status}")
    if res.success:
        return -res.fun
    elif res.status == 3:
        return float("inf")
    else:
        return 0

In [ ]:
#The below function calls all of the above functions and will return the max h_m or inf

In [ ]:
def H_m_computer(G, n, k):
    tuples = Tuple_generator(n,k)
    h_m = max(LP_solver(G, *tpl) for tpl in tuples)
    if h_m == float("inf"):
        return float("inf")
    return h_m


In [ ]:
#The below code below will create 500 samples of the specified n, k, and m value

In [ ]:
def generate_dataset(n,k,m,num_samples_per_config=500):
    config_dataset = []
    num_inf = 0
    max_inf = 0.2 * m * num_samples_per_config
    while len(config_dataset) < num_samples_per_config:
        G = G_generator(n,k)
        h_m = H_m_computer(G,n,m)
        padded_G = pad_G(G)
        sample_data = {'n': n, 'k': k, 'm': m, 'G': padded_G, 'h_m': h_m}
        if h_m == float("inf"):
            if num_inf < max_inf:
                config_dataset.append(sample_data)
                num_inf += 1
            else:
                continue
        else:
            config_dataset.append(sample_data)
    return config_dataset

In [ ]:
#split the below up so I can run them one after another even if the system stops
# I am running Jupyter Notebook in WSL which is limited to only 1.6GB, because of this I originally had it run and append to one large dataset.
#since that took over 20 hours and was only halfway I broke up dataset generation and saving into the 21 different combinations of n,k, and m.
#Then I combined and randomized the data

In [ ]:
dataset = []
dataset = generate_dataset(9,4,2)
with open('dataset2_1.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,4,3)
with open('dataset2_2.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,4,4)
with open('dataset2_3.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,4,5)
with open('dataset2_4.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,5,2)
with open('dataset2_5.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,5,3)
with open('dataset2_6.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,5,4)
with open('dataset2_7.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,6,2)
with open('dataset2_8.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(9,6,3)
with open('dataset2_9.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,2)
with open('dataset2_10.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,3)
with open('dataset2_11.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,4)
with open('dataset2_12.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,5)
with open('dataset2_13.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,4,6)
with open('dataset2_14.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,2)
with open('dataset2_15.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,3)
with open('dataset2_16.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,4)
with open('dataset2_17.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,5,5)
with open('dataset2_18.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,6,2)
with open('dataset2_19.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,6,3)
with open('dataset2_20.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
dataset = []
dataset = generate_dataset(10,6,4)
with open('dataset2_21.pkl', 'wb') as f:
    pickle.dump(dataset, f)

In [ ]:
#I stored my Gs that I created from my code and switch them to only be Ps and flatten them

In [ ]:
def transform_G(sample):
    k = int(sample['k'])
    n = int(sample['n'])
    G_mat = np.reshape(sample['G'],(6,10))
    P = G_mat[:,k:]
    padded = np.zeros((6,6), dtype=P.dtype)
    padded[:P.shape[0], :P.shape[1]] = P

    return padded.flatten()

In [ ]:
#From the data that I am using from the dataset that keegan created I need to reformat P to fit my model

In [ ]:
def transform_P(sample):
    k = int(sample['k'])
    n = int(sample['n'])
    G_mat = np.reshape(sample['P'],(k,n-k))
    padded = np.zeros((6,6), dtype=G_mat.dtype)
    padded[:G_mat.shape[0], :G_mat.shape[1]] = G_mat

    return padded.flatten()
    

In [ ]:
#Below loads all of my datasets and splits each before combining them into the train val test sections
#This means that for each n,k,m have the same amount in each train val and test

In [ ]:
files = [
    "dataset1.pkl",
    "dataset2.pkl",
    "dataset3.pkl",
    "dataset4.pkl",
    "dataset5.pkl",
    "dataset6.pkl",
    "dataset7.pkl",
    "dataset8.pkl",
    "dataset9.pkl",
    "dataset10.pkl",
    "dataset11.pkl",
    "dataset12.pkl",
    "dataset13.pkl",
    "dataset14.pkl",
    "dataset15.pkl",
    "dataset16.pkl",
    "dataset17.pkl",
    "dataset18.pkl",
    "dataset19.pkl",
    "dataset20.pkl",
    "dataset21.pkl",
    "dataset2_1.pkl",
    "dataset2_2.pkl",
    "dataset2_3.pkl",
    "dataset2_4.pkl",
    "dataset2_5.pkl",
    "dataset2_6.pkl",
    "dataset2_7.pkl",
    "dataset2_8.pkl",
    "dataset2_9.pkl",
    "dataset2_10.pkl",
    "dataset2_11.pkl",
    "dataset2_12.pkl",
    "dataset2_13.pkl",
    "dataset2_14.pkl",
    "dataset2_15.pkl",
    "dataset2_16.pkl",
    "dataset2_17.pkl",
    "dataset2_18.pkl",
    "dataset2_19.pkl",
    "dataset2_20.pkl",
    "dataset2_21.pkl",
]

train_nkm, val_nkm, test_nkm = [],[],[]
train_G, val_G, test_G = [],[],[]
train_y, val_y, test_y = [],[],[]
for file in files:
    X_params = []
    X_G = []
    y = []
    with open(file, "rb") as f:
        data = joblib.load(f)
    for sample in data:
        X_params.append((sample['n'],sample['k'],sample['m']))
        X_G.append(transform_G(sample))
        y.append(sample['h_m'])
    nkm = np.array(X_params, dtype=np.float32)
    G = np.array(X_G, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    part_train_nkm, temp_nkm, part_train_G, temp_G, part_train_y, temp_y = train_test_split(nkm, G, y, test_size=0.3)
    part_val_nkm, part_test_nkm, part_val_G, part_test_G, part_val_y, part_test_y = train_test_split(temp_nkm, temp_G, temp_y, test_size=0.33)
    train_nkm.append(part_train_nkm)
    train_G.append(part_train_G)
    train_y.append(part_train_y)
    val_nkm.append(part_val_nkm)
    val_G.append(part_val_G)
    val_y.append(part_val_y)
    test_nkm.append(part_test_nkm)
    test_G.append(part_test_G)
    test_y.append(part_test_y)


#for sample in data_records:
#    X_params.append((sample['n'],sample['k'],sample['m']))
#    X_G.append(transform_P(sample))
#    y.append(sample['result'])

In [ ]:
#loads keegans set and puts it into a dictionary

In [ ]:
other_data = joblib.load("results_dataframe.pkl")
print(len(other_data))
data_records = other_data.to_dict('records')
print(len(data_records))

In [ ]:
#splits the data by the nkm

In [ ]:
data_by_config = defaultdict(list)
for sample in data_records:
    key = (sample['n'], sample['k'], sample['m'])
    data_by_config[key].append(sample)

In [ ]:
#stores the split data into a bunch of pkl files bc I was crashing whenever I loaded too much at once

In [ ]:
for key, samples in data_by_config.items():
    n, k, m = key
    filename = f"dataset_n{n}_k{k}_m{m}.pkl"
    joblib.dump(samples, filename)

In [ ]:
#below loads the datasets from keegan and takes 10000 values of each and splits these 10k into train val and test
#again evenly split

In [ ]:
files = [
    "dataset_n9_k4_m2.pkl",
    "dataset_n9_k4_m3.pkl",
    "dataset_n9_k4_m4.pkl",
    "dataset_n9_k4_m5.pkl",
    "dataset_n9_k5_m2.pkl",
    "dataset_n9_k5_m3.pkl",
    "dataset_n9_k5_m4.pkl",
    "dataset_n9_k6_m2.pkl",
    "dataset_n9_k6_m3.pkl",
    "dataset_n10_k4_m2.pkl",
    "dataset_n10_k4_m3.pkl",
    "dataset_n10_k4_m4.pkl",
    "dataset_n10_k4_m5.pkl",
    "dataset_n10_k4_m6.pkl",
    "dataset_n10_k5_m2.pkl",
    "dataset_n10_k5_m3.pkl",
    "dataset_n10_k5_m4.pkl",
    "dataset_n10_k5_m5.pkl",
    "dataset_n10_k6_m2.pkl",
    "dataset_n10_k6_m3.pkl",
    "dataset_n10_k6_m4.pkl"
]

for file in files:
    X_params = []
    X_G = []
    y = []
    with open(file, "rb") as f:
        data = joblib.load(f)
    for sample in data[:10000]:
        X_params.append((sample['n'],sample['k'],sample['m']))
        X_G.append(transform_P(sample))
        y.append(sample['result'])
    nkm = np.array(X_params, dtype=np.float32)
    G = np.array(X_G, dtype=np.float32)
    y = np.array(y, dtype=np.float32)

    part_train_nkm, temp_nkm, part_train_G, temp_G, part_train_y, temp_y = train_test_split(nkm, G, y, test_size=0.3)
    part_val_nkm, part_test_nkm, part_val_G, part_test_G, part_val_y, part_test_y = train_test_split(temp_nkm, temp_G, temp_y, test_size=0.33)
    train_nkm.append(part_train_nkm)
    train_G.append(part_train_G)
    train_y.append(part_train_y)
    val_nkm.append(part_val_nkm)
    val_G.append(part_val_G)
    val_y.append(part_val_y)
    test_nkm.append(part_test_nkm)
    test_G.append(part_test_G)
    test_y.append(part_test_y)

train_nkm = np.vstack(train_nkm)
train_G = np.vstack(train_G)
train_y = np.concatenate(train_y)
val_nkm = np.vstack(val_nkm)
val_G = np.vstack(val_G)
val_y = np.concatenate(val_y)
test_nkm = np.vstack(test_nkm)
test_G = np.vstack(test_G)
test_y = np.concatenate(test_y)

print(train_nkm.shape,val_nkm.shape,test_nkm.shape)

In [ ]:
#loading the saved 221k data points that I sorted
# I have taken the nkm and P and made them into some variations of those as well as G

In [ ]:
train_nkm = joblib.load('train_nkm2.pkl')
train_G   = joblib.load('train_G2.pkl')
train_y   = joblib.load('train_y2.pkl')

val_nkm   = joblib.load('val_nkm2.pkl')
val_G     = joblib.load('val_G2.pkl')
val_y     = joblib.load('val_y2.pkl')

test_nkm  = joblib.load('test_nkm2.pkl')
test_G    = joblib.load('test_G2.pkl')
test_y    = joblib.load('test_y2.pkl')
train_P = train_G
train_G = []
for i in range(len(train_P)):
    train_G.append(train_P[i].reshape(6,10))
val_P = val_G
val_G = []
for i in range(len(val_P)):
    val_G.append(val_P[i].reshape(6,10))
test_P = test_G
test_G = []
for i in range(len(test_P)):
    test_G.append(test_P[i].reshape(6,10))
train_G = np.array(train_G, dtype=np.float32)
val_G = np.array(val_G, dtype=np.float32)
test_G = np.array(test_G, dtype=np.float32)
print(train_nkm.shape,val_nkm.shape,test_nkm.shape)
print(train_G.shape,val_G.shape,test_G.shape)
print(train_y.shape,val_y.shape,test_y.shape)

In [ ]:
#create the inputs for the DNNs
# create an initial preprocessing DNN
# tried to put CNN here and it made my model worse

In [ ]:
inputs_nkm = keras.Input(shape=(7,))
inputs_G = keras.Input(shape=(6,10,))
x = layers.Dense(448)(inputs_G)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(256)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(128)(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.1)(x)
dnn_in = layers.Dense(128, activation='relu')(x)
#reshaped_G = layers.Lambda(lambda x: tf.reshape(x, (-1,6,6)))(inputs_G)

In [ ]:
#A function to create an expert model from what I tested when I had l1 l2 regularizaiton on all three it did not work as well
#if I had the dropout at .5 for all three it did worse and if I had it less than the current values I saw overfitting
#when I messed with the number of nodes they would either reduce or make things worse
#the input is the input_G and the output 
# I noticed that I had severe underfitting after adding what I put above so I removed regularizers and minimized the dropout
#this allowed for my training error to catch up to my val error

#I tried to create and test hyperparameters of the values below but I could never get the values that 
#seemed to work there work for my actual model so I decided to scrap the couple of days I put into that


In [ ]:
def expert_builder(x):
    x = layers.Dense(256)(x) #, kernel_regularizer=tf.keras.regularizers.l1_l2(0.001,0.001)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(64)(x) # , kernel_regularizer=tf.keras.regularizers.l2(0.001)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(96)(x) # kernel_regularizer=tf.keras.regularizers.l1_l2(0.001,0.001)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.1)(x)
    # x = layers.Dense(16, kernel_regularizer=tf.keras.regularizers.l1_l2(0.001,0.001))(x)
    # x = layers.BatchNormalization()(x)
    # x = layers.Activation("relu")(x)
    # x = layers.Dropout(0.1)(x)
    out = layers.Dense(1)(x)
    return out

In [ ]:
#I tested with a few different numebr of experts and noticed that the more there were the better
# The layers below the experts_tensor will take the inputs for nkm and then get a softmax that selects the different experts that are good
# These softmax values will be dot producted with the experts to return the true output.
# I also have some callbacks to speedup the training that monitor val_mae to plateau the learning rate and stop the training early

In [ ]:
experts = []
for i in range(5):
    new_expert = expert_builder(dnn_in)
    experts.append(new_expert)

experts_tensor = layers.Concatenate(axis=1, name='experts_added')(experts)
flat_G = layers.Flatten()(inputs_G)
#p_feat = layers.Dense(64,'relu')(x)

x_nkm = layers.Concatenate()([inputs_nkm, flat_G])
x_nkm = layers.Dense(512)(x_nkm)
x_nkm = layers.BatchNormalization()(x_nkm)
x_nkm = layers.Activation("relu")(x_nkm)
x_nkm = layers.Dropout(0.1)(x_nkm)
x_nkm = layers.Dense(30)(x_nkm)
x_nkm = layers.Activation('softmax', name='gate')(x_nkm)

final_out = layers.Dot(axes=1, name='final_output')([x_nkm, experts_tensor])

model1 = keras.Model(inputs = [inputs_nkm, inputs_G], outputs=final_out)
callbacks = [
    keras.callbacks.ModelCheckpoint("mheight_experts.keras", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor='val_mae', patience=300,restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=50, verbose=1)
]

In [ ]:
#take the log of the results

In [ ]:
train_y_log = np.log2(train_y).reshape(-1, 1)
val_y_log = np.log2(val_y).reshape(-1, 1)
test_y_log = np.log2(test_y).reshape(-1, 1)

In [ ]:
#compile the model

In [ ]:
def log2_mse(y_true, y_pred):
    sq_diff = tf.square(y_pred - y_true)
    return tf.reduce_mean(sq_diff)
model1.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss=log2_mse, metrics=["mae"])
#model1.summary()

In [ ]:
#below runs and trains the model

In [ ]:
history = model1.fit([train_nkm, train_G], train_y_log, epochs=1000, batch_size = 1024, validation_data=([val_nkm, val_G], val_y_log), callbacks=callbacks, shuffle=True)

In [ ]:
#Save the model so I do not lose it

In [ ]:
model1.save("dnn_mheight_experts221k_P3_1.keras") #add graph for loss and mae

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

In [ ]:
loss, mae = model1.evaluate([test_nkm, test_G], test_y_log)
print("Test Loss:", loss, "Test MAE:", mae)

In [ ]:
#print a bunch of random points from the test batch of samples

In [ ]:
pred_y_log = model1.predict([test_nkm, test_G])
pred_y = 2 ** pred_y_log

random_indices = np.random.choice(len(test_y), size=100, replace=False)
for i in random_indices:
    n, k, m, _, _,_,_ = test_nkm[i]
    actual = np.log(test_y[i])
    predicted = np.log(pred_y[i])
    print(f"Sample {i+1}: n={n}, k={k}, m={m}")
    print(f"   Actual h_m:    {actual}")
    print(f"   Predicted h_m: {predicted}\n")

In [ ]:
#Function for running the inputs and outputs for the tests

In [ ]:
def recon_G(padded_P, n, k):
    padded_P = padded_P.reshape(6,6)
    P_top = padded_P[:k, :]
    actual_nk = n-k
    P_sub = P_top[:,:actual_nk]

    I = np.eye(k,dtype=padded_P.dtype)

    G = np.concatenate([I, P_sub], axis=1)
    G_pad = np.zeros((6,10), dtype=padded_P.dtype)
    G_pad[:k, :n] = G
    return G_pad

In [ ]:
def transform_P(sample):
    k = int(sample['k'])
    n = int(sample['n'])
    G_mat = np.reshape(sample['P'],(k,n-k))
    padded = np.zeros((6,6), dtype=G_mat.dtype)
    padded[:G_mat.shape[0], :G_mat.shape[1]] = G_mat

    return padded.flatten()

In [ ]:
def m_height_calculator(inputs, outputs):
    final_nkm = []
    final_P = []
    final_result = []
    for inpt in inputs.keys():
        n, k,m = map(int, inpt.strip("[]").split(','))
        for i in range(len(inputs[inpt])):
            final_nkm.append((n, k, m, m, m, n-k, (n-k)/m))
            sample = {"n": n, "k": k, "m": m, "P": inputs[inpt][i]}
            #print(inputs[inpt][i])
            #print(inputs[inpt][i].shape)
            P = transform_P(sample)
            G = recon_G(P, sample['n'], sample['k']) 
            final_P.append(G)
            final_result.append(np.log2(outputs[inpt][i]))

    final_nkm = np.array(final_nkm, dtype=np.float32)
    final_P = np.array(final_P, dtype=np.float32)
    
    model1 = load_model("dnn_mheight_experts221k_final (1).keras",custom_objects={"log2_mse": log2_mse})
    pred_y_log = model1.predict([final_nkm, final_P])
    pred_y = 2 ** pred_y_log

    return pred_y_log, final_result
        
            

In [ ]:
#Add inputs and outputs below and run them

In [ ]:
inputs={
        '[9,4,2]': [
            np.array([[ 12.34182835,  78.8825531,  -74.04528809, -93.71873474,  76.5475769 ],
             [ 70.00410461,  31.84829903,  10.89520359, -13.96429634,  51.48582458],
             [-34.13206482,  38.65733337, -17.05139732,  87.81211853, -51.16646194],
             [-52.76997375,  -4.83906269, -59.08993912, -43.87838745,  48.50744629]]
            ),
            np.array([[-20.93441391,  16.07522583, -74.35280609, -92.421875,    31.05505753],
 [-67.70335388, -10.42372894,  71.59892273, -92.68517303,  33.64113998],
 [ 81.48065948, -49.71315765, -43.87363052,  75.6060791,   35.69181061],
 [  6.75473022, -63.70702362, -19.96879578,  17.10869789,  11.18667507]]),
        ],
    }
outputs={
        '[9,4,2]': [
            354.5741951135599,
            703.001648767227
        ]
    }
pred, result = m_height_calculator(inputs, outputs)
# print(len(pred))
# G = G_generator(9,4)
# sample = {"n": 9, "k": 4, "m": 2, "P": G}
# h = H_m_computer(G, 9, 4)
# print(G)
# P = G[:4,4:]
# print(P)
# print(h)
# G = G_generator(9,4)
# sample = {"n": 9, "k": 4, "m": 2, "P": G}
# h = H_m_computer(G, 9, 4)
# P = G[:4,4:]
# print(P)
# print(h)

In [ ]:
print(result, pred)